#Import libraries

In [1]:
import re
import nltk
import pandas as pd

from nltk.stem import PorterStemmer
nltk.download("punkt")

import pandas as pd

%pip install gdown
import gdown

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer
)

import zipfile



[nltk_data] Downloading package punkt to C:\Users\Probook
[nltk_data]     G8\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


Note: you may need to restart the kernel to use updated packages.


# Load Covid dataset

In [2]:
# Download dataset from Google Drive
file_id = "15VdFMp8JfCdjxOVA062qiykMekD__Adh"

gdown.download(
    id=file_id,
    output="Covid.zip",
    quiet=False
)

# Load datasets from ZIP file
with zipfile.ZipFile("Covid.zip", "r") as zf:
    df_train = pd.read_csv(
        zf.open("Corona_NLP_train.csv"),
        encoding="latin1"
    )

    df_test = pd.read_csv(
        zf.open("Corona_NLP_test.csv"),
        encoding="latin1"
    )

print("Corona Train Dataset Loaded:")
print(df_train.head())
print(df_train.shape)

print("Corona Test Dataset Loaded:")
print(df_test.head())
print(df_test.shape)



Downloading...
From: https://drive.google.com/uc?id=15VdFMp8JfCdjxOVA062qiykMekD__Adh
To: c:\Users\Probook G8\Desktop\Assignments\COMP723_Assignment\Covid.zip
100%|██████████| 4.60M/4.60M [00:00<00:00, 5.35MB/s]


Corona Train Dataset Loaded:
   UserName  ScreenName   Location     TweetAt  \
0      3799       48751     London  16-03-2020   
1      3800       48752         UK  16-03-2020   
2      3801       48753  Vagabonds  16-03-2020   
3      3802       48754        NaN  16-03-2020   
4      3803       48755        NaN  16-03-2020   

                                       OriginalTweet           Sentiment  
0  @MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...             Neutral  
1  advice Talk to your neighbours family to excha...            Positive  
2  Coronavirus Australia: Woolworths to give elde...            Positive  
3  My food stock is not the only one which is emp...            Positive  
4  Me, ready to go at supermarket during the #COV...  Extremely Negative  
(41157, 6)
Corona Test Dataset Loaded:
   UserName  ScreenName             Location     TweetAt  \
0         1       44953                  NYC  02-03-2020   
1         2       44954          Seattle, WA  02-03-2020   
2

In [3]:
print("Training dataset shape:", df_train.shape)
print("Test dataset shape:", df_test.shape)

print("\nTraining sentiment counts:")
print(df_train["Sentiment"].value_counts())

print("\nTraining sentiment percentages:")
print(
    (df_train["Sentiment"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nMissing values:")
print(df_train[["OriginalTweet", "Sentiment"]].isnull().sum())
print(df_test[["OriginalTweet", "Sentiment"]].isnull().sum())


Training dataset shape: (41157, 6)
Test dataset shape: (3798, 6)

Training sentiment counts:
Sentiment
Positive              11422
Negative               9917
Neutral                7713
Extremely Positive     6624
Extremely Negative     5481
Name: count, dtype: int64

Training sentiment percentages:
Sentiment
Positive              27.75
Negative              24.10
Neutral               18.74
Extremely Positive    16.09
Extremely Negative    13.32
Name: proportion, dtype: float64

Missing values:
OriginalTweet    0
Sentiment        0
dtype: int64
OriginalTweet    0
Sentiment        0
dtype: int64


# Select the text and category columns

In [4]:
#Select feature column for X and target colum for Y
X_train = df_train["OriginalTweet"]
X_test = df_test["OriginalTweet"]

y_train = df_train["Sentiment"]
y_test = df_test["Sentiment"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (41157,)
X_test shape: (3798,)
y_train shape: (41157,)
y_test shape: (3798,)


# Preprocess the text

In [5]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [6]:
def preprocess_text(text):
    text = str(text) if pd.notna(text) else ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

X_train_clean = X_train.apply(preprocess_text)
X_test_clean = X_test.apply(preprocess_text)

print(X_train_clean.head())

0              @menyrbie @phil_gahan @chrisitv and and
1    advice talk to your neighbours family to excha...
2    coronavirus australia: woolworths to give elde...
3    my food stock is not the only one which is emp...
4    me, ready to go at supermarket during the #cov...
Name: OriginalTweet, dtype: object


# Remove stop words

In [7]:
stop_words = set(stopwords.words("english"))

# Keep negation words because they can affect emotion
stop_words -= {"no", "nor", "not"}

def remove_stopwords(text):
    return " ".join(
        word for word in text.split()
        if word not in stop_words
    )

X_train_no_stop = X_train_clean.apply(remove_stopwords)
X_test_no_stop = X_test_clean.apply(remove_stopwords)

# Lemmatization

In [8]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return " ".join(
        lemmatizer.lemmatize(word) for word in text.split()
    )

X_train_lemma = X_train_no_stop.apply(lemmatize_text)
X_test_lemma = X_test_no_stop.apply(lemmatize_text)

# Display the original and processed tweet text

In [9]:
# Compare original and processed tweets
for i in range(5):
    print("ORIGINAL:")
    print(X_train.iloc[i])

    print("\nPROCESSED:")
    print(X_train_lemma.iloc[i])
    print("-" * 80)

ORIGINAL:
@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/iFz9FAn2Pa and https://t.co/xX6ghGFzCC and https://t.co/I2NlzdxNo8

PROCESSED:
@menyrbie @phil_gahan @chrisitv
--------------------------------------------------------------------------------
ORIGINAL:
advice Talk to your neighbours family to exchange phone numbers create contact list with phone numbers of neighbours schools employer chemist GP set up online shopping accounts if poss adequate supplies of regular meds but not over order

PROCESSED:
advice talk neighbour family exchange phone number create contact list phone number neighbour school employer chemist gp set online shopping account poss adequate supply regular med not order
--------------------------------------------------------------------------------
ORIGINAL:
Coronavirus Australia: Woolworths to give elderly, disabled dedicated shopping hours amid COVID-19 outbreak https://t.co/bInCA9Vp8P

PROCESSED:
coronavirus australia: woolworth give elderly, disabled dedicated 

# Experiment 1: stop words removal, lemmatization, max features: 3000, vectorisation: TF-IDF

In [10]:
# Shared feature extraction settings
def make_pipeline(classifier):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=3000,
            ngram_range=(1, 1)
        )),
        ("model", classifier)
    ])

# Naive Bayes
nb_pipeline = make_pipeline(MultinomialNB())

# Neural Network
nn_pipeline = make_pipeline(
    MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=100,
        early_stopping=False,
        random_state=42
    )
)

# SVM
svm_pipeline = make_pipeline(
    LinearSVC(random_state=42, max_iter=3000)
)

# Using the previous fine tuning hyperparameters
param_grid_nb = {
    "model__alpha": [0.01, 0.1, 1.0]
}

param_grid_nn = {
    "model__hidden_layer_sizes": [
        (50,), (100,), (50, 100)
    ],
    "model__alpha": [0.0001]
}

param_grid_svm = {
    "model__C": [0.1, 1.0, 10.0]
}

# Tune and evaluate each model consistently
models = {
    "Naive Bayes": (nb_pipeline, param_grid_nb),
    "Neural Network": (nn_pipeline, param_grid_nn),
    "SVM": (svm_pipeline, param_grid_svm)
}

results = []

for name, (pipeline, param_grid) in models.items():
    search = GridSearchCV(
        pipeline,
        param_grid,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train_lemma, y_train)
    predictions = search.best_estimator_.predict(X_test_lemma)

    print(f"\n{name}")
    print("Best parameters:", search.best_params_)
    print("Best CV macro F1:", round(search.best_score_, 4))
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

    results.append({
        "Experiment": "Experiment 1",
        "Model": name,
        "Best CV Macro F1": search.best_score_,
        "Best Parameters": search.best_params_
    })

Fitting 3 folds for each of 3 candidates, totalling 9 fits

Naive Bayes
Best parameters: {'model__alpha': 0.01}
Best CV macro F1: 0.4298
                    precision    recall  f1-score   support

Extremely Negative       0.68      0.23      0.35       592
Extremely Positive       0.76      0.25      0.37       599
          Negative       0.43      0.55      0.48      1041
           Neutral       0.61      0.41      0.49       619
          Positive       0.38      0.67      0.48       947

          accuracy                           0.46      3798
         macro avg       0.57      0.42      0.43      3798
      weighted avg       0.54      0.46      0.44      3798

Fitting 3 folds for each of 3 candidates, totalling 9 fits

Neural Network
Best parameters: {'model__alpha': 0.0001, 'model__hidden_layer_sizes': (50, 100)}
Best CV macro F1: 0.5646
                    precision    recall  f1-score   support

Extremely Negative       0.61      0.58      0.60       592
Extremely Positiv

# Experiment 2: stop words removal, lemmatization, max features: 5000, vectorisation: TF-IDF

In [11]:
# Shared feature extraction settings
def make_pipeline(classifier):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 1)
        )),
        ("model", classifier)
    ])

# Naive Bayes
nb_pipeline = make_pipeline(MultinomialNB())

# Neural Network
nn_pipeline = make_pipeline(
    MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=100,
        early_stopping=False,
        random_state=42
    )
)

# SVM
svm_pipeline = make_pipeline(
    LinearSVC(random_state=42, max_iter=3000)
)

# Using the previous fine tuning hyperparameters
param_grid_nb = {
    "model__alpha": [0.01, 0.1, 1.0]
}

param_grid_nn = {
    "model__hidden_layer_sizes": [
        (50,), (100,), (50, 100)
    ],
    "model__alpha": [0.0001]
}

param_grid_svm = {
    "model__C": [0.1, 1.0, 10.0]
}

# Tune and evaluate each model consistently
models = {
    "Naive Bayes": (nb_pipeline, param_grid_nb),
    "Neural Network": (nn_pipeline, param_grid_nn),
    "SVM": (svm_pipeline, param_grid_svm)
}


for name, (pipeline, param_grid) in models.items():
    search = GridSearchCV(
        pipeline,
        param_grid,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train_lemma, y_train)
    predictions = search.best_estimator_.predict(X_test_lemma)

    print(f"\n{name}")
    print("Best parameters:", search.best_params_)
    print("Best CV macro F1:", round(search.best_score_, 4))
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

    results.append({
        "Experiment": "Experiment 2",
        "Model": name,
        "Best CV Macro F1": search.best_score_,
        "Best Parameters": search.best_params_
    })


Fitting 3 folds for each of 3 candidates, totalling 9 fits

Naive Bayes
Best parameters: {'model__alpha': 0.01}
Best CV macro F1: 0.4336
                    precision    recall  f1-score   support

Extremely Negative       0.64      0.24      0.35       592
Extremely Positive       0.71      0.25      0.37       599
          Negative       0.42      0.52      0.46      1041
           Neutral       0.61      0.40      0.48       619
          Positive       0.38      0.66      0.48       947

          accuracy                           0.45      3798
         macro avg       0.55      0.41      0.43      3798
      weighted avg       0.52      0.45      0.44      3798

Fitting 3 folds for each of 3 candidates, totalling 9 fits

Neural Network
Best parameters: {'model__alpha': 0.0001, 'model__hidden_layer_sizes': (50, 100)}
Best CV macro F1: 0.5743
                    precision    recall  f1-score   support

Extremely Negative       0.61      0.62      0.61       592
Extremely Positiv

# Experiment 3: stop words removal, lemmatization, max features: 3000, vectorisation: CountVectorizer

In [12]:
# Shared feature extraction settings
def make_pipeline(classifier):
    return Pipeline([
        ("features", CountVectorizer(
    max_features=3000,
    ngram_range=(1, 1)
    )),
        ("model", classifier)
    ])

# Naive Bayes
nb_pipeline = make_pipeline(MultinomialNB())

# Neural Network
nn_pipeline = make_pipeline(
    MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=100,
        early_stopping=False,
        random_state=42
    )
)

# SVM
svm_pipeline = make_pipeline(
    LinearSVC(random_state=42, max_iter=3000)
)

# Using the previous fine tuning hyperparameters
param_grid_nb = {
    "model__alpha": [0.01, 0.1, 1.0]
}

param_grid_nn = {
    "model__hidden_layer_sizes": [
        (50,), (100,), (50, 100)
    ],
    "model__alpha": [0.0001]
}

param_grid_svm = {
    "model__C": [0.1, 1.0, 10.0]
}

# Tune and evaluate each model consistently
models = {
    "Naive Bayes": (nb_pipeline, param_grid_nb),
    "Neural Network": (nn_pipeline, param_grid_nn),
    "SVM": (svm_pipeline, param_grid_svm)
}


for name, (pipeline, param_grid) in models.items():
    search = GridSearchCV(
        pipeline,
        param_grid,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train_lemma, y_train)
    predictions = search.best_estimator_.predict(X_test_lemma)

    print(f"\n{name}")
    print("Best parameters:", search.best_params_)
    print("Best CV macro F1:", round(search.best_score_, 4))
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

    results.append({
        "Experiment": "Experiment 3",
        "Model": name,
        "Best CV Macro F1": search.best_score_,
        "Best Parameters": search.best_params_
    })


Fitting 3 folds for each of 3 candidates, totalling 9 fits

Naive Bayes
Best parameters: {'model__alpha': 1.0}
Best CV macro F1: 0.483
                    precision    recall  f1-score   support

Extremely Negative       0.48      0.52      0.50       592
Extremely Positive       0.61      0.54      0.57       599
          Negative       0.45      0.45      0.45      1041
           Neutral       0.51      0.58      0.54       619
          Positive       0.46      0.44      0.45       947

          accuracy                           0.49      3798
         macro avg       0.50      0.51      0.50      3798
      weighted avg       0.49      0.49      0.49      3798

Fitting 3 folds for each of 3 candidates, totalling 9 fits

Neural Network
Best parameters: {'model__alpha': 0.0001, 'model__hidden_layer_sizes': (50, 100)}
Best CV macro F1: 0.5721
                    precision    recall  f1-score   support

Extremely Negative       0.61      0.56      0.58       592
Extremely Positive 

# Experiment 4: stop words removal, lemmatization, max features: 3000, vectorisation: TF-IDF, unigrams and bigrams

In [13]:
# Shared feature extraction settings
def make_pipeline(classifier):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=3000,
            ngram_range=(1, 2)
        )),
        ("model", classifier)
    ])

# Naive Bayes
nb_pipeline = make_pipeline(MultinomialNB())

# Neural Network
nn_pipeline = make_pipeline(
    MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=100,
        early_stopping=False,
        random_state=42
    )
)

# SVM
svm_pipeline = make_pipeline(
    LinearSVC(random_state=42, max_iter=3000)
)

# Using the previous fine tuning hyperparameters
param_grid_nb = {
    "model__alpha": [0.01, 0.1, 1.0]
}

param_grid_nn = {
    "model__hidden_layer_sizes": [
        (50,), (100,), (50, 100)
    ],
    "model__alpha": [0.0001]
}

param_grid_svm = {
    "model__C": [0.1, 1.0, 10.0]
}

# Tune and evaluate each model consistently
models = {
    "Naive Bayes": (nb_pipeline, param_grid_nb),
    "Neural Network": (nn_pipeline, param_grid_nn),
    "SVM": (svm_pipeline, param_grid_svm)
}


for name, (pipeline, param_grid) in models.items():
    search = GridSearchCV(
        pipeline,
        param_grid,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train_lemma, y_train)
    predictions = search.best_estimator_.predict(X_test_lemma)

    print(f"\n{name}")
    print("Best parameters:", search.best_params_)
    print("Best CV macro F1:", round(search.best_score_, 4))
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

    results.append({
        "Experiment": "Experiment 4",
        "Model": name,
        "Best CV Macro F1": search.best_score_,
        "Best Parameters": search.best_params_
    })


Fitting 3 folds for each of 3 candidates, totalling 9 fits

Naive Bayes
Best parameters: {'model__alpha': 0.01}
Best CV macro F1: 0.4374
                    precision    recall  f1-score   support

Extremely Negative       0.60      0.28      0.38       592
Extremely Positive       0.72      0.26      0.39       599
          Negative       0.43      0.50      0.46      1041
           Neutral       0.60      0.43      0.50       619
          Positive       0.38      0.65      0.48       947

          accuracy                           0.46      3798
         macro avg       0.55      0.43      0.44      3798
      weighted avg       0.52      0.46      0.45      3798

Fitting 3 folds for each of 3 candidates, totalling 9 fits

Neural Network
Best parameters: {'model__alpha': 0.0001, 'model__hidden_layer_sizes': (50, 100)}
Best CV macro F1: 0.5618
                    precision    recall  f1-score   support

Extremely Negative       0.58      0.58      0.58       592
Extremely Positiv

# Display the experiments results

In [14]:
from IPython.display import display
import pandas as pd

results_df = pd.DataFrame(results)

# Round the scores to 4 decimal places
for col in ["Best CV Macro F1", "Test Macro F1"]:
    if col in results_df.columns:
        results_df[col] = results_df[col].astype(float).round(4)

# Fill in missing experiment names if necessary
if "Experiment" in results_df.columns:
    results_df["Experiment"] = results_df["Experiment"].fillna("Experiment 1")

# Display the comparison table
columns_to_show = [
    "Experiment",
    "Model",
    "Best CV Macro F1",
    "Best Parameters"
]

columns_to_show = [
    col for col in columns_to_show if col in results_df.columns
]

sort_column = "Best CV Macro F1" if "Best CV Macro F1" in results_df.columns else columns_to_show[0]

display(
    results_df[columns_to_show]
    .sort_values(
        by=["Experiment", sort_column] if "Experiment" in results_df.columns else [sort_column],
        ascending=[True, False] if "Experiment" in results_df.columns else [False]
    )
    .reset_index(drop=True)
)

,Experiment,Model,Best CV Macro F1,Best Parameters
0,Experiment 1,Neural Network,0.5646,"{'model__alpha': 0.0001, 'model__hidden_layer_..."
1,Experiment 1,SVM,0.5507,{'model__C': 1.0}
2,Experiment 1,Naive Bayes,0.4298,{'model__alpha': 0.01}
3,Experiment 2,Neural Network,0.5743,"{'model__alpha': 0.0001, 'model__hidden_layer_..."
4,Experiment 2,SVM,0.5512,{'model__C': 1.0}
5,Experiment 2,Naive Bayes,0.4336,{'model__alpha': 0.01}
6,Experiment 3,Neural Network,0.5721,"{'model__alpha': 0.0001, 'model__hidden_layer_..."
7,Experiment 3,SVM,0.5557,{'model__C': 0.1}
8,Experiment 3,Naive Bayes,0.4830,{'model__alpha': 1.0}
9,Experiment 4,Neural Network,0.5618,"{'model__alpha': 0.0001, 'model__hidden_layer_..."
